In [ ]:
'''CASMI26 | Molecule Finder

Two rankers, one candidate engine.
Public reference: megayak/CASMI26 Two Rankers One Engine (0.337 public LB).

Candidate evidence combines direct and adduct-shifted spectra, mass-shifted
same-polarity analogs, spectrum fingerprints and MetFrag-lite fragments.
The final order blends a fingerprint-aware ranker with a robust ranker.
'''

In [ ]:
'''Check the competition data and every public asset before expensive work.'''
from pathlib import Path
import glob

REQUIRED = {
    "test": "test.parquet",
    "train": "train.parquet",
    "template": "sample_submission.csv",
    "COCONUT meta": "coco_meta.pkl",
    "COCONUT fp": "coco_fp.npy",
    "fp bit map": "fp_bits.npy",
    "public ranker": "rank_train.npz",
    "simulated ranker": "sim_rank_rows_nofp.npz",
    "fp model": "fp_*.pt",
}

for label, pattern in REQUIRED.items():
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Missing {label}: attach {pattern}"
    print(f"✓ {label}: {len(hits)} file(s)")

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

In [ ]:
'''Materialize the pinned public engine locally.

The embedded source keeps the Kaggle submission internet-free. It writes pv.py,
pv_fp.py and casmi_engine.py into /kaggle/working for direct inspection.
'''
import base64
import gzip
import json

PUBLIC_NOTEBOOK_B85 = "ABzY8000000{`uOTXWmUlIHvV3WPe&3Xn;Pq%O9pX2-~~+;-Tq)Rua?XD9|6BtZ#H5`Y1alI-@0dD__LosHQ4uy6Y`@3a5p{F2Qt>w+o}l+=Ar`)T^bu?V29Sy_3jtjy0FqhL16Hafk}{A-ZkCBa5#WA3NZaeNbP)Hdcp?vMT4-{^e)QrpPlMLI%>Z=9pU^OHC2edjml;wE-J_|Y^-vzl`n1<rAF6-L2jbQx`JUA#MZ`C)6z`P;w#FDJhaoMbVag(Jt0{8@ZePtrK?ulziWBPR)yU`F+vJoUpUbHY4xQf}6vuYY~kdhTT5d@=L$VC>A}Suk46f~?_OT!)zx#d$D{<EfJd!$mk7XWZG%H5GFL9!^cy&YZ2ScCEeFELw3UX*_qHI<s(?`e_(sG-5aKN7qh%6OMw8<7bZV9L@YJtG8(IWAjDZ8Tnb@)f|63M%(mfG0N-NbvVILMVoid+|NhXVRW^%RpS;Wm>@=3OwymtoOnVrZ#c)xAic{`-boh(jq4x{oRHwK`_!4!Un+fc_G67E?tFjtV*|kc=9_PvpRWDf0X%{my!y>Ky7r^1fL7_J>pS}QB=ECEN~3c3>XUdjcCONRk<jzefAgH*s6xG7*Z)U-v9L}W-$)>BZ9R29E@*I07SIH?of0Ssa<qDR;EypCUbf%NY2FazfKc@;_tPcy{F?^j$%hY^R}j%_zaAy^;MSifGhVP}W1o=iZ~y&&I`nIImwMshZEYP^Lz@;moY7L|cUa0am=OG;QQ)X@!#jbFDUCYK;?x<36IsS{o;dTsCm7QjI9U>m2qWoT(a0!XM7f++=FbB?d(;!~rCI*cJ9+Zn??3s)S&E0(_&p7RL?Cw*EU4j`GmEo~=J=b&FTepaezw<^Gk*584YW~!VsPy4Dsb%WDR3O}wxX5sQ^MUe$nrECVY+!tl`zzL5^J!e0Zl&(qMSD1l1C5)SNyv^kH-NoGwlWE2Dr!Qi-}!fcjitKXJJuPLFqRqn-dn-Xqri0S|b0(PXp@qB*<T-{#AVz=7HnB_qX3}zkj{EeKgyi#3^+{@V)j^LfVm^QvYd2m`S+jU(IQ#8L+B=ETActAs&Z|d3FEPA%OwdG|HWEP#-T6qCre`XgJg!8kmFxJEy(pxVgW`<9U$QM}8DXRB-0JiH@${ybbPJyBNz~Tg|_q+R^FJ>H8lqsIR*WW5z~XL}Nl49?>XHgLIR=sAj@UP&&*eAh>3;nCA1F+PjC&Ebyn!O`PV}&NPZ|u=)LAyvQ%3i-L60K(B%{Ny8{-*w1Kv!mI0Ct`k#FS|{pf9@9GM4Uwm`2!xwNGS4wINy9Q~ZpI*wB5I4OT?eyqoq9FMBhBZI&;<lhA?NOMrjaXqwdPy}(Sn9ByW^G|YRPkk)EsI(-T&La{u|+Q8-S0Tt0>8~=W*n3&$Dsp&$j1;;6ox5+sSN^O$qYFuoVlTX8mR0Y-K?ZZPAN33mODFNnVWUXAA5B5H5miN;H%<0k(9Vnk2x;MOUy?5NOai&tv*(aTSo@ocNsRB}4Oi^9Rdt46@JxKw2Krlw}s)`4KHWQJVzRRSxb+#Tq)ihMD6szr$#KtRgk8*P#l?Sq#z-gz*GBaoZk&I(44bi7snm8xnjL_ssQ|Y%!dNSw?8t7-h>}z&ogLD5{20T~ju+=nQ88x&qB1%HM44?0E;yFeZEzXjGT|PC{S}CUQfxCJzW&am-BWh!Brv+TkGy&g>8)nqlHwTMBerTLio<NzKj*Az`an5<<P4KA4og<^X*(D-bdU!OA2!H6mOw(T7Rk7g&`Bi;OFbo#2*6I}XMKRkgj_jiw1a$p?Q?W8EIbqc~dRL@u&<*xm=(7>8pB8o$&e6{r)5a0p9$#Y4w5;6--rPw4;Y)>dQVOU)L+MuZhB;#Y7>YXLY6m;-Heq6N{q4O-!3kwc8@8zNdbPl%E@F_7gB+j15U>E~idLf0tB=uIBZNlLpIoL?M%xNy4s+`vDscNyJW&*&9#Mgpaax()o}Zf;NgD_ZI8Frq%Ux3;#^@iffq?PhzwvD4VuYEZ?^nj`os@rUk3x(GbN&Ghk&t!v~m3MVAg%{rHnL;n?XY0w6Bdo<040Fh6KT5~q*b82Lh1WWt@*CN({pY@OYZ#|0ZAuaZ;ei{>RrU&Yx*X;Lev`>Q3wEN1Rk$gv;3yAU<4;)4Yck{!`w_%R4;%#FvAOeHkG#F5YaWDbqjNORngeOORjg4Lb*~uz2KBL~3RX~8f-F0a%dZI~!3hfJSxkDQ@%mU{X?ZEeO{)+Hqe4M6nD#jqnrvc%9#N}z8v?)(d&S*$+)=2#Px-kw@6n1&&n}o+Xy-!IR$TzVDV&bS%Llt6cIo047_)M=x*Lb1lLmX{N`)8AA1(t~Wpw$S{+-(x%SBkA{hDYcl-#9JjKmOf0*TjH8loE-{SS%+NLO(#9z*E)Uw<*JAbi6`Q9QR%55IOaUR1$`_yb_5Kf|lySVZ389V(@^SRz=5PZ$x~#`hZ&-fcF35xc6xf9=H8VaGT7C%MF;e3`B)r{tOcGm%o5~2cYIovTP)GM8P-kKQ+C3^|(<?`z2zT&6RvlQSX`~w+X~3kdAuudA+qy;2p(D7>qB8<E2AtZ%%s=QVmI)SqR_~-<!n{$)q+%ZDb4;@RPTUNCl4oaSe!9qZ+2fvR=9%^t+5mC=F$8mTuE2LHiey!d|{Se)&s7%~bDtT7YZ)5H03}CnP^b$z{YOo!$(6M}7V^%xR5NNQ7W|m(d84yL0sFdr1V&&Q1nDy?IZYuGMTbi+8UM-@SWrc=VP&?C1|~-d_w}y*oYqKyT?~vn^hq9R55wJ3f4SPOsa0`|{<#3@%RJiGr=7`q}AE$Ee;?Z64FhpN`*r{~ClJL`A$le1G`v^m{q9c2f`G{Xl%7x6kCO^EW5D*&S8xr`F)?!<*9&7!!#JJG8$^SZGSRbAEjMQhcd7tr}^L&W`-@{fiTNKPF|xo%^=~qQm)hcOSd$c0d@IcK4c8hvhihNQ0EPYnKMMQ-d-*$b?ihpzo9JGZbtO(%@==P5g;*q)QW*U0s}>@#OZz%6#M0|J47`L$*SdzDB?Q1T=I<B(w{*K|+t8<-lXXMFf8uj0kFxJO5O3fV7iY9H%52#K{z*5~&n564Ua5&@mJbiwJ<keyEngft)U3T}LsOC}{-qB)`L2({F0m&v)8hZQTda&iq!Ou6QDS&<SDn;GN_~s(aaEnD@fI^Bq;{h>H9|4~SlU`}dD1zfP%T28**rZ5}uy4Dt62u!}}IRPPCGKsmfI#;ZFXamU0~_jid&{`v<yphkPY_5la9VRSa3zQ5x&HL5RH2Vl><qW_WCahB*(z?%9C`~9MHESe<c9ny`v^m-$wF8Vb}P1|PA0@L*D*LIC^)cdq8yQjWmyZ33k!QB!B1$d`zu~?`oiV>k+JLv-_4j%$Zb`N%r%(aQ<Vwm$!3Q7kI@B(V21c?2GensIW$wyeWrCzp-maW;jcR57mZ4xMTtzy3#RnN_#KQw&VkZ2o2pz8ImeBJ_Fi0ixj=vfGP?tBYoP-^Oi`)==%0?dE5oO7CLyF8+MXQ+GMa&EZt6ZvLk)b!1U+M}S?wnXE2PO@z%IK<=Vs8mM>MC;v;pr#)al4kH@lz^Vj6nueXO}WVrG$o}fgn@JUG}46i{IHC2x9iEl$E0_w*6R=pCSK?T<44u-i+dWLml_@!4ZlFcBfH^|+3=xk`G97fo1F|76LW(_u(KAMj7i_HAbmI25IaLcsE7r;U+LG}7}l7Ie#HLj6OAE&5$UtWJf`w>f_n%3_J+V9ntw<eF=X0^&I%fd|43{_HBd^k7ESKgzM`mAwX&ex2hB{@Qhim`HfH#^L5d0j5RV#PX<3Ja@mLcofZDeOuf$u6S+2v3p;uxdq9JD+O<3?>Dnw+2Sr*8$w6#!@%xD2!SXs72X+~Z`G>=-9b#_^Nd&F)ep~X*0%@1bg<`jySu`dasXoF8CHJrKNE&T~2-lvb1fnZl-Z2lCaapv;2rDnYfwgDJU0Yjk@6lG0df>YsTZ%T{lgWgRhy-{E63I0u3)v|GXh(+^Bw1}fTjFb@1t<6DrsHT+fc$z)GGwjQ`%HI#n)#^=9CQr(Ft9=qgXz<kM8d_M8LYvmY!2O2^t<apS$!$#>^(YAY>xcEJJTCAM)FlM2L*Ei7e&W@)tbw)kKJlesU<nqAgs(vbg9oXSv@tU&GK>UGdm=iNlylwavvolAF$<q&Fk}Md43)OS`fW!Y5%Ce1k7KYp;|XjUc}xnio8p*Cy^C?`WgP3_lnw`!gs?Mo{C8|hY5kxP6tkMeHRn34IVtbZ6t7b#C#%a^+Jx635f0RTa0|+k<^}|cpvT@UCg@M)&NJTpp8Z#~7s}|G>K#e5lfQ~7QuQ_aL@%N~+YyT?OeK6uQ=C<XnR({vilf`&*t2dF)o%fnZyq?^^!15puFPL+k2c+TOohW>Tpz_5OhYh*|3lCQMzenmS-&5#{*$^t3Zo4g|9C9sDG*FxNCt6HXUU`x#`JnJ^K(IzRuM*`1h+8YGA=Np5(j>tWdqH~fzxAV;sWH>){4rS&yxX|K(AjshsLwUk_yp1Wjb?{5wIRwW1vdf@9-bS7`@_EUmm^wJo<*8BtbL=795E2>o<u}7Vzi^Vkm%xanqwjpejw93q}2V_3f%M4}*>rZ8?IDF)j>@r^MThl)QV&gc|A+Ls6z?X@FnDMkr4yzD8#!hv(<Y+IrM&k>p)-Uc*A#eEO`t*W770pVplB-4+Sm&7J1c-DY$5S<N}^w)Ps&pYJ|z?e4dp@7J8O?oPAu{OQiq{nq|Y%~rS1yE`rVMlH2qxP8@aJ)^qMcB%0;{8+j>yN&1j&z|kI_o?X@Y4<68Z#~=Dc}hF$4UK8P@w~aa`?TEdd%xRmH=aLx{&fF&`&sjO&3W71p{n~kPn-K_>OAbWpEmgO{@zY|j~ef`Ftz79&Anav9(12Rr>^$*+ASELkEKnWK0e!f`usU*t!E!jFHT`12Aq1EuQ#D!9v?Wb+o!M_la^9^cp`tFZW0c?f4!?~z2}d+X5j<p)#-<m!;52G=!grITP&Wr)x)FX3sI@~aJ1TbMa7pde>}RtN`DqWyxb5@NS7NO$8E6*Ud)xa8h@h+SYI(An!5h%TVLfweE}b@W_0qlqTpLqj*HX@nw4XrGW1E6YTr`~)8(Q@$L-hswk}MVhBDS1t)}YcXi8tyIE==RUhkZ~t{%;kdlyD8O1ZhC_v@M~uk}u4lke>&>#zHDa}w6fF~{lx7RYH>?bCLtphDcb2BtZW_Uql18VYOcB@{cFS;gcOXFMmh8deDlu?QcEkF>n?mGvbgwF;H-)U2S}Z5P|3y|OLZ2I99X5nq+kyQgjM9<!gf6)nBB8`6+$udZUdygu!R)~CI)K<#@LsBHtV{YU`r6ad^=0pLzCqaAxjm(dr&U?Idwone3jc0P_09C1Y;pDboGR~|mGun_qX2^0QU9zJA#n)-M09KGg@aT?x@hrg0gX~+|P&2dR6q;AF@L<>nQ8(|*IGuM<bAjr6Aon&!aVkUtpx^rPgCee5fd4k4-A6w2Rg4CGpSL=k|KVPCKdmRw?Ti(~`x^!54HaLHCq7I4Q4_+M~;>>xcRV@7gLmfnMI(KjNdfR77^;=rwv}|O#_3*YA_f=u7fQc^x>M5<m+`k1m5T9sqTAtDUr=nX(`nQcNAG@~N;@(oF+XnoCT@MSJsq>juP=79LEJ{Nd%))6vO%^@y{M0(IayXbseU%CiYtoN`U&^1PUDbqd;D8=UsqVgy;l&O|SpKOs7NuHJ?M8O(CjtBPunI9TK9OckJHfRi@DZjvnl!S-+-)g!XK8PwVvs9>`D{sG6{n+q?#`ACK$)O})8LVSiMAud9Hl1d2;WKv(_q^gmZg}tWt71bbj8X^_%wPHvZuj)MtP9(44r?%_X|Ve@uBI70^5!|%6gp|j|YqhE%pSsJD_=@*b`oAqxl;FsL?D;T$Az`3uv7)H$@Ld3qJHa7h~CS+#8&2@n-RK=N^O9oRi-ws@&XDZWv_wK+KZ<m7j4WlN4_;=Vo8Lq~Yq7ydexSngY_xO81Xdz#s553-D{t1|J`ql;FTKGl#8*Y7C&z!po<(#UwP&`H6|2e)0f*n!C+JQBza*ZxlWmD{fX?bC4dT7#XVbnMU@JXj;Sr3g-l#5!Hi>iV1bl3m6cHJW)NuAA?Ma7ig}If5pUVpk&mMU6fy}9)>zD&!t;b_bPT*zjC?s(AKR6Sm6GHaOJHZjm(CzTaB?l8qlVB0w^T*wksG2W;a`kbr&S-JEtj@ziu4l`B!NWXlxP3QuDOO?e#@vZu+~fs%@QK`<4CriRf3L;`)9E{%X&D(X)SF&$nthW;kWj_2C$vlwK(L3G2oER=peel?by^j@o8$mf8@r0AY;NbQA{@0zL_S)PE0tUlNo{!DKB4O0!ynaksw@<G>`>?BGdivo{o7=lr5Nout-|ngEg$*v==Z*g7Uf;liI#sCp`s$4q6{5<VD~v&9@XPJc9Ydj_^It)UT`+l!$Vh)|<09!T(=@!722@7%Gu<bGl`N;q7?l0-C0=v5aFW@Qqy6cw*2B3{!p+-^w{ZL@RpvpoF;jPcm=JhS8$)1jKPHIaWAOJsq56*>EQ!nf^N{O9+0c=96*J)1|H9Y<L7HechXZPeR*k9W3RZ}SxIY}4N68NI8t?romqi)q{2e1-a^c@O$L6*4V+n=ev&Wg7W5-|$FR?R-!)J5I^UM>;aKXd3!9&qEZmZGD?B(5PYV+dL1@ueSJ+zWJKqM9^YGN+F@o<JsUU^^<Ein}<pOWzSxA>{p(>N7n&Kbst{7i?6PNRFqOrjhA5t2XK0FH`DQ9k~Fq9m0u%uin@)Hc=n1~J)h%-4EI&ANl+}NR%+klp*WV9H@*+@_lx;q9z#WyE+LjoWGcAOYPpImZ;+syo{tZ2#zeaM0vwI2fFvZmJ-?d|;~At0kYh;0czY4&{){?*O)bIhrbq3v5CLK$bm5hu594SIHKXSa(a?+FDvU5B?wv>lULHqd{mSFv5B2b0puX$bHu{^IXt(A>h8|aVoI|&l1;%NdXfJ><+^%}PKKC&&&lHSfnvh7OPxaX_5JZc40Dohb+bcU@q6m7ZIx(Z(dVXJ>p$yqb@oR6W-zakeZn9a+h_S~r=w~#ctaON!t@r;zL%pnDH5O(ZXelFkUzU@e-_*?_6`u?gVOJxrRq~`;;PMLIH<CDUy;b+UB?gF@F8UQtc=E0a4`JEws2_SPqG<`CrN@IS9*yHMfJUEBEmB)u=dT53sN17#ToU_E_?8dTz@KK_wtT`0cXxIbSg)|_EW)7C{%U2j>g7|8<_3;ZJ6Cce59I-)z^1}`ddS-d*QDs@cg1kb_JzdKyAc|dQqhEE?6Z9PrP^`W^4r445k6M$n9t#tvNI_4(eTG(SLuhNzUa*}$d?8u<ka?huT^_2s=%*BflwZ>u9SP!*lW(OU#coTW*TKs%S&3g!s^-BBr$1_`IEq88F_lASF8f-l=^!+!u{l9{u9PgH4@pU<o2Ct9E{863G#RBdr@<eAT@kDq+5ooZ|lU-?9Sn<j!$r9Cy&7}jt)P3cys(=-8qr4)z+$xZAzcomEVje3j?|Qy7quceXp{#7dCXe)Z5i_C~bGHug+rKw~~EbeSP2vWL7R-KG?8K-BlO3T#Z#&Mt?eh37mhBLaB`5>{)?}FSI?R3ryB^>o^H%v3&{0BHHgnI(6uVJxyZpT6W^u*EY0bNr(oSMIwg~59Ap1;-ZJk4O$FA@$t1+Z+H4a5h@_7h(s*e99wi*sfk9!StD@Qo*pIvY<vwF175Ve-|3M$P!ldl0FANl*@BQVtSI%afS}NFJmr-v$>k$FVD()M;dvx!0!>CQlp@5_vdfVAE*W}3n&Zg?6L5LiOlmbXY>MdkiEw3U3ez=ps*Kl~q)naQghD&a@EyJwbv-aliBQI(Zi@KDQ%U`V^RBlBKOaC$m}KBP@D)*0;E^d4*$83^MRBBLgi8F9sI071hJi-z^2eYqR^>mj@E?SQf9&#an=k*e<Je-(aZy%6Ea2j@zLfjw?0t|Q=*Su68~ySo|0EF+HnRG%{fpzu7zdf^2IfoB4X%Q5@lKQ{dc7p2fxbA)Fu2LQKmT?vPDhFhx*8Pq?^YpUNCXTAPlikJ6Q5MvEJkD@gdv%-G{G0_1Nd~pS6J%7c}zIic=Q$7uX4%qzA1l=A()`4NeD#|gCHuN8<Rye!V>#4e0l|%`|509UgmAJpDFL1!Miu-I4$iJGMVkGV@Xb?d{ma#R)n;|A7<6qTL@2OfoWAps;zo~SiT;GPe*k>%4er$dU(`Kw+!12qvUR3IZf0|FN@0V+N)!4R}9|tgD5+e*e*kO`gRdbzq{PfK39V`KSP@KJEZnVgMx3{5{%V_#`fm@`Naogt@Ci6U7WrWt)HgDFeeREJpG{Gzqt^9<NP{Exp&3C;i;bvCjLB}kq%?<+-yZ_m4@b=mHLzlH)e1XHCWOn29w8=P0S!K7*kOomID+ez{EE{PIBXElFu{a@%nC<hGWMn66V$;$7%|fL8;T0F{+gjwT+LZ(R?fw-LY8FZUH$`o{|CBmT*DA!7msrsZ=cXK*!~;U31GyKih6AsPELAn=tC`H6aj@Ha5W1GqnQqPxjdZnzXLJg}Gfo_HUH2#HSSzsZbwRuQ*qI5Up_#Ep}c3qn49}4&K;P=1aaEF7C6UbRUo6*Pk&Ud6`?Bt=xB?`;SjH<kPg`TKDgt=XV7*nEuG=I7mybE1~IWJzR{YK`z0)y$y6RPb$i{phpoKcfP=AH#4MO%dWzmmP{T)zMF@>_&Omwi9|W;aYIOZE7xX6&bJC<J18nI7XlBsb0tTIMpZGU!dw!}R__$x<r-iWP0+tR8)Q9t)%SWGUC+Gg%{Mi5<KM1rJQYW>h3PCT0ZrI&VUVOZ8Nmh<B~dlZa@X_vB`8Hf053D$8;;cEAXdQ1mk6kP-0V_Nvh-bgTjK%tI%v0Vd0ZHH2skhr;zy-#lotKXYO$yk%PLDlF`**cdVATHulc8nQ?r`0Tzi%#0-KzN(eZrfl5}$tj~6rTo5WUmQhdUEFaVfHhZw^xW)^qrd`NX2c6S<Wc{hQBbQmj`EfR2|hHk<1r(g&6Id)Lnuv+^|b{v7IMNPUbro;NO7wz=ooA@;&UYYo#K!zlVJUJE|d{7Mw)hCTKxFXD;O)^~I?vuM2M$1j=08WvRej7;$J%)zC;87qToV)SUv4p^F`O?{KV-aN^7eVkTAXN;b<j$yV!|_C{j~Fx|k=qkLNhfiAPl=Z!B6_~pht6%S&x=_+nm%rMXie-hlj@Z?5M#Wi;a``46v2;&`g-Tz1?hVnAdgk?kJF|67DlATy3`vSoAvj&{B>MjfvCOOc&W%_B8r>`1k)%F{h4dExywD)P~`jLcR#w~=S!M0=~*sMy=IJpi(@3^eo#{MWYxWA;c=mDY{K<L&3RvQUJ9Q>xb7V|cY+!MjO;?Fihd*|t;X9}LDIeR8qh9X(F%zkVoA^-Z42?>HE8AL3nDme*58E(uE}B8zxA&em~w$r|C&KNfTyn-*~p&-<H0z_h0PfDzJZVMe$Z!JIYN?k^E`s3W!jw^{eY*#lAvMdiY0gb28nC53*#(cw{DNN83RL3?4@UdUT%bnmL?Ndq6oY{E&OLS_fhyC4cWD5+}&#tz-GX*41tKQ1Am-#pNXx}Z8lnuB~)UNpjM@K^G}0d4lXT;=PIht9w?AZt?dFLG!{oEt3@Hy-QSvUppF{!F&K|Yfo9v0si;)68;YxO<ywV4Q&s6|W^Z#`Z>T6X1ksYsi*<Vkw%ilMY%JGA607oou2?yXgA1U>?1*<i1wq62(2%yE2<_(Xv&WN>HalA|dU^NI2o;Bk_M^}{VETt5LK7VY!rWS>Ak6Ru{V*6Lfb61$!n3tb<-7)MiFxAANii{{z(i4&Wa=7R#z0l!aBrZd8_hh7)@EOV`xq@>sg|#1D=jO$<bhUtot2yxM*;EI76OxbcDDjFrFRduCxw9cs+EW#lvwdL$7hpMQY6P8yF@M36D<wtX<3QV856xS+Qc}aRhYKZgpqB9vkMjG1=DMBi9x&{a{SV9unb2*m+CgePmgh?)U@`bFg8?&$2W8(H?oJV_lU)NMl*Jc7w*Ca=g6jtci>YCF2KU9TLPo)N{l+UvI69$Y-)H8)02EK@U*o`|MfmKQECLsE4t7)sPuXo6`bG)!aVCD1ZBoR)PDbL+pS4^E-P9!t!(kGrFVr>3jmwseCoh$nK!+sdNt9jOT}O~NCt>o$g$`)N67+FMuqWtfKyo=W3*5;Ajb>7HJbJ|c?g?*a3(yC@l$dqJd2;3v;cYRVp@%0Ni@rDkjDcT!aNx8<xkXPh-{j4(v>iPAw~RT>j}(5G#7B^WCn?O^OF8U=+x~^TEJu$<}QBr>MeNyC}8R-#gRr}jz_}On4G1F*b5>lr{S-K*V0l0%dfU_37a&(>gX%wGQ6GFa1LiPN9b;&csBDB9ES5nbND0E(_$QJW4^X;GlHN+c4HE*LGluj*j@ZhDt*4!x1zGjp{yS8*H&Fdl!3@n6wRvz>HtY~KhWK)6;vjqbxjO)6XH^g_SRh~AoatV@Y@#*)|{~RKUvX65AM@Mr2&R&5XiIQCRgcf%J2t~_jWi;ihThKL7$kb^?t>HR9}W?*;@@|Mg#bjBjw=-m+kOKR7Kir*nxIE5!y)<D;3fD@~WMPL)l1ke+=ItzDp#i5FFPBL&S1*&uZ9{f(jL^UYxMA0e6+*3w%6V#cRi(r9=xkum=K;AHb4RPvaYh*EHkix22;2HSKM458~%2*k;ygV!K?VUT8YR4N4AiXWlxTNA(I<Km-}?oguo7w!#>;wHMMe>WuK`y)d)5qHzn`&q{km>>$<^ywa7BFn_Q}qV5ecGO8jaTwLQ2cxGJ_{M4gj9SIFwXF1?#fAZas-wnCLQb7Z&VE!krHyr!UZAUT@KO4|CrmY2zE!jROy!2?KT?dAGAvV#48|M56ti2@uTv#X4jBf@g1ol?f5w<gZwjrEBYL0%<+vF1v;v$9XX~k7cc?nbW)c5Y|t8{`<I6hj&ao!}4YE|833m!dZ<Hku<ofU^Mz8Tny+%2p~AqVkml2wW2(-R+xx?(yJPlqS~B6!?}S-0s~?#LW+N8WXD)v+=Y-rq$n^ii*QcLt8SBE`u>A8U)FVbG1@Fq?{qnA1`wkcE(2LdXapL|((-kgK%mz$89`?kJMevAVO_ZHfI8#<Mf#n1kmQZjZf!@Q5nLmSl2coakT+6YK=|#fGLwEW@WTaShOVhLHOSmX9*wR-=0a>9!aaHD=yQ5&nomI{M@rCmlTLPL(})MeH&3h~fet<Ks9a)Z^)R>tUkx&ABz{Va244r6AEbnL|L9PG1}>Uu;_WyS>M1Q_mO=q6`BfUMXfMb+#5L!|HuTMf9fOK&8l%0g)3e{4o!b9e9wa%vJAXU@e3J5-E~0$iF@WcBBz0EaN)a1Lu`+T$ME9Z~x=}h+HJNfn6jqsX71ge_Kr_NpMMza}PKO>_f7c%BO?-fieqEIYE!dM~X#C4w=Ox)7dN>q2bk3`VS6US0wmwkVQrnh5rfQKD^6=I&FwLesIh${*oh+l(6vyBDlJq4I&jGhLwoJ&M6WcIJl-H&vP!2D~ugEQF2b6Gxw2xC&>P69~k!98Hk)BhAYf*G_L1y9f?cANQF9c%c0_j@%WC^gOd=s5;7-fwg(El)E4RbL359<MB&K>x5J0kvpZrVNOJ>0EECXDX%pbuWMolr#v$!Ws?l&hAh8Ssa(0ItDMXFwJLgs#ygNNQd`G%3=S?}!{i`4=LcZ0%E2AF05eP*;D$xWVtlUd*lu@w(=(j`~`c>e}N*Mf7IuT?GA*>8XQzq8z02F79AX<hgVH{0{xS7A0<?iLi>Dlr77w-<w2k(D88C<;naD4dkTzD|FE;qa?$01lKqA`Nakpv5S*c~zD_(sDKuRrS7{OId;lBEpIpDgw5ombxjmS?9NAz#R)g^~Ef;rq9e=KeG|lmYDba1CZ}&lA+@8>zH|1CC_2t|AeE+!@Fdl%yfhNO^%(gunl~f$uM)i)J~)3*(na`=Qngg5BkYfqt|wIMC-Mx{N-Yo^m{)UP^^JmzO`%rq3=fU;k8pcYJ(#`7+3+d7NBcas!teeUN6+3lrzw<%X5w#PV5L$&&&moyOv1mOD21q0FIDBu`PPOMZ<*#BPgopw-qE(2Lt#NugqKAelHt8Xkma<u#uJs5~n=q{0K)RSJ94l{?R_)LcZ$l*#2r@3XN4^^0a_ztNn0$@<P`D*Xk+oy-=Qj(7SMZY!b#NmRZ<U)mNYyJSn8EwFx2EEn-AHUF(it4#$L?(iqtz9+ambO%pGKQ(f?{dK3YNBhhYFL6Yed~BQvLW1QKBa!w#jw=ais-bujM0b8GOYqrCbs+eODmxI8Lpcm2j*$rhx_}!Z?Fp{;W?kAM^Q=9Xf68ccc=FaS>ksjKNPLZ$5@}kQW!R&kJdt6;8%uw-2$WD5PbSv2GBK;F;DsDp(2C71gYp|uHV|a+z9I_Kr7(<h^cl%GQv|6EwL#VVzuZwEOiA+(fir&?oMAM&4yQ=nCwL8v+x{&46pRNMU&0ixg&-?_>l-w*4&n_FQb|wQf?VZWxc)z5No(@K7zDd`LG%lF$xWG`a~CKaFj`r~2ACQuO0VnYK6HxNXg6vQ@*u%yBaa7CYrV%yNkz?!*)RNn{l`J^Vm1cXo!Aa2B0^av%?>R)BJIj91@aa>0&PhMOzun*c@J%MxZ%FlVi?&i5=8!<7CDY48f$g<Pox4~at}1KR;OZSKpmyL7>i+Jl=r}Z%FZ?fY7=<lG4{LBy24#rR;o#atbx3r(G@~^P*2I7aAz_`8EbDYZE`W%Tfz1R8ZHId5(0zLXIQk;ymnd|y16?aJ$%p;yP1kj{pi-Px<u9vcwxKF)W|Rp%pBZs`80x#JSqs>s)e1^>;qn#aul-45M;Chq9Dg?E?+32ubzmM)++6~XNp5X*H4xWD`=Xgy|&WK!X=?|m+x@QKXqkX+%WIT(734Bl@`HyH<~l;?t*?-+~1vcO&i>_G(D#FW(mG(YZdu=a!pA?>gtq^#RUj6v7_hwSf#T&u2G1TVXT<<6~;ffV$V<Ge)}Y02NQRFd;&W%LGitT-~*0MVEC$-(Dy+EYj%83FoVkt8O=cN<-rwkRz8s1kfGQ3sFP**%MBVPzBM9J!Z$p#%DgQqRF43+{ZHcb%8&FbH}%Kif-h)Kl5^VN-QDK%eXGfxKiK5X+9uQKLrrqh$5&@Xo3bV0<UzXI8Dg04n|Fqj0ph6fK7BC4#J#I`a(Hxfj+Dcdr5dD!jnXj5p?=C5$FNVF#Iv7r>yTz})iF@&9}}kn*PjjdgQVgtvOokt(tz+`_I@!xyL0Cz&08?hdq~_WFySBECZobXu1(Et^&(Q+5kI`82Y0S{scy3G^~~MYU-5*Xl8HZ>;ya<Cr{XUai%zYsyLP_PSYZ?4xa!DUJ)mL!{-7vjw4{-o=P>4^W7;SMC9>cD9vZ}Vk8_eQl>0X0GeCuxV*lE$H!fsm?_+s{7wda2Hr_tC3k-;<au@L%$TGByZZU|)k<R`730alAUZ=I&*PBQOLesYEKekT8MK~J|z~kI^HF<L{v({_AnB}3o%Lncz;eQ6U=x5Feen^+Jb%|gvH<0KU*9<MXK4M9P4S<3P#0eAu(;hR1+QXHhO*fM1jH=^ZIyBlr(*`eD(?ZcbqN9lWybo4FsnV$90cUNuBVEpE-I%g&^h+`ecP8}Mas?%D_A3Rw9G5|oapR?*`>#>}^D{}kQv?fm*Au_`7$>!&-to`=7mr<ABi|K21}bYkA)=^f_K>H&zvgKpoHjyugySZIoT8`BbzqX3Ty7{4=rbD(WSLIw%R!~+11pPhmy#>xDaz~qktkdskgG5cC^b(ktOcm&h<BK{1}m)bX(6@Pz}`Z>WvHXzq<hU~={~bO8BVQ@!$v~36$IKJ1qa6xlVrFw97<V5u>-A!`nZvu8t9~Y>$M)NDYu#Iaw+>{T&;=0p!K3@iyKPptf}7n`kK<+n&S9^;#h2<-{KB6x*!;?sl!6JfU;FbhPjDjskVgK1kwPJD%{DhbnnqR3u=jU$nX$q!4-B&bfQa)7zc$YOgPf(aPu-^P2mQ@O35VYP7?MHLNi_b!%vuE7ys~+*kE1Ej-KVH&ahS6p&<0Mzs`d+q@7_00ggR&%GNzAwl+b^s$D$v{#yKQaJ`1!bz`@*kyxepEV`lnVC)*bH6*b<Gwv5sB|?Y{x9SyvF8KH&(wy9&GB~d2L8`_GP2FpJYZ&tSo;(q4u^)eD|M^a{U+VaV=7gIsH9mkeGGmp)3dk<8AWWZf4kL*h{mC3YvB5sm?^!U(^<;;SZ|6y=ofH<`GSrHt<dgcZrZn=pcT_m?I`3OC9ewyF?Wig{R_U1bo)cZ~E1B2_$le86fYaD?Oq(@<6^WZ%tV=wZU_bOwN+S1oQo*n2JD8J5twWfvI2CNJjFsE~TUQfiZS?1hXEZpTaNvxApgnGn+tB411<(aNVX{#W;T}P^CsIJMNvy`f!TMF4Ffs5Z5nbVkQ1pZl;K_iEB^Yls=fP@^vl3|&Wh=?pfoz@r)sBpIb~0g)u`&@MB2xio@f-F(`7_M<=h3V08#=?ccr<PntFN4CTRax2yDRKG)M=MHW*3jHy-Mt;C}!EU;URLZb@i0GYFgbiG$X?ZNt3sT`%Z^ypO(0XZrgMItK*tu;1ITI*y4;uGn9B*9J7^H#u%)?w2CE?$<7sTp}orI9bHbW1U`Z{_Qb^(#ZunT2im!UDcC!xC_tOKqCoE*f12qL!Uozh8zu_y*m`wcQF0ndkv{9ZRA4tIEjm~sL0tAtZ<FS;+3#)A=gqz)wMm?L^VP^|wM&9qxQURcX?a~}7dZ9iYY|T~A$o~N(+-!}xYpd_SNG2~kIwCHYwQDm9}Q5OQw5$E$xf-{55#d1HBL`s7FRIo57?hf6a<gv8pP|-k4WS&`03626@GsmKDw<Z6>Zr(-xrbFACbtBkL#W9Fcr8r{a{|h;|!e~q+U+J8cuH(cdo-Vae>VnGau*3W(R@)hms?CF-#S>-RCg>gclXrEbU~LJQl?Y_(Adl{%tUWlYbdpoW4`}2<`o<CX^KOsqv@^EI>$QfeD$5456ivTS+NGL+?Ju&zwGC{S?14QcC%Zb$I7W7LIjS$5xrRF<`O!f}cx9zLs*U>}ya6;g30<2ob#30Z>#<k-l0zl6qk*n<#yzx|9sJRpnW$$O>rj8_Tmc<MWTKym@}WA-2rLSA7Mf|2v%Le^_;%=MH{Yb&nSwv?XH&zmnd_2Rz==cY5QQp(}De0g+o{i!tr&>;&6UNaFWU2rw|gJ#7lXX&yCMIH)EG)s>&9$~GMs!QrFps5<O$p)p+)SwN+e;S3g0_;C?f5=IYAu$TjMB2i@-q?zUPb7EZzP`3<zz=@kDZkjB-@-=~zbEA^T?{Z>ZR-rW(`+a4VJF7`HX@F`Ia==uBf(a>7r>mA(HL1d5B}qj*?&T=!tkjPYHl-O`(!YkEyC=P0`@AL~HYPPz7f>^CONV70den<_KPArECNS)LuU{E1kA@M5_Ify-8-P+X74coD;4qBX-__;s^DrV*xyrA*5+B!s%DM$p74U}k5f*~B%?Z=zrY1ZKPZ*v*@h5Rnq3YZD`g$~cb)sEk?6qZ(LZmi@5y|o+UnY66xRq=TZl+o38YW)?39lmI)5nnuaW_l~<+NovkfD^dt-E4BFlqfLxI0nznR5F7L6CZ$8oi}Wi4N>Eyg!tsfHhOapfJy3qC}T<i0T9eI{lOtKoG6~CLGG%0GXgh=DREtX+@Gux>lA_n97_|HQ9k-2l9y;4n~2yG$R(<aUNtCNBqf&-DUZpH8us_r4D1cdK@BC?dU{Vn#YN8;#$CC&p1R`^}ny6qLdS+k?2!IjNF0QDqf_@>AS4C6yD)4-kl!3J^zZPo>{S&rQBW13VNXAz9oyTMw*YMR~`F@NVmHz@|<8fm!6fdMLQ^{L|p{H%pJ*WW>%syW`L8#vreL^(CP(Q&iJucm2lf<Te~);_7ZU&Tiy?(#wt0>flP;1nMF(v$Icg)sDxqVWML}Pn3XdO+1JF|`!k1?vxbR-hD_bB&D0$3O&+$IIBaKK(l9^SFPGh4S9VRl8OAX`LCAw1HD{-A;bdX=WLsgg(QG;x`dQ%OX4Ire$G|Kg1(S$;_6C3C^2%&-<#SWggY4s*2cTNk?MrF;fWN7bM1^e<FVKg0{fSVW-{S$%g@ym7IcIkFu;-lcHLa3myoV;>t%MVPr1Zi8GV>5NqHWtRXdSh@zAi%$byHJ>T--J)9%N|0N30pm{I)cHEwG5784^ANdR@4$-iJo20vH5O?ulc*&#vXm-U#86(OaDZ*D?Kh#E=EUD-Bu>sDeI8<P$cyQd3yd+v=9qPK~2X`{VM=MDXci%02y3^nB061UupA{2SOqP{h@KVt-;rp^nBk*2rkx)I0oU4`(^cjsYgT=UJ-P{P5-Laok1pwTyc4JXz!`TEDtio9k;QbD>F#bLqe^cWA>wIM#*_)+#m9kIJc9uAuwR!*LBas!Ez6`mth2C3by%1%|B)iy<=8ruhQbecW<-3x8m<{j?!{#Y<Z3u(DjWm-w2TXl&m=Gt}p|>bhy#FddwM-)%ELofu+Gbng#((?K5(_*;qv2^c65*_Bnc=^tDA7!JgxWc^-8ZKp4$`Zdogykr4ObGXV$eAVfim#rbZtJP;jOt5cR6g|<uJ@XiZo;}N7YoxDwh2t*c1!#V^q`rys+|@vuXs)#I(wMS(?H6TBs|HlJ&RZC%Y*+7!S(d%IIZq`#lN;q2qGhE+B+t3e*!Fvtf`?yHA@!{qU_;kF<K}WT)67*JqZ4(NHRciRygGdM?#1EJ+wuqkTktT{tRg28ri-XEA#PR<;5&MCHwcz;*8O{SoW9aX5|RA{LI@){Jfj|n8t_PP-*Zk;^&eBp`!`ChD8zTf7n~Ib#VSkxj?PEG2}X?)#1_;iCp(;h>(Za>n*I^O3*<Rlp^ve8pK)X5)-!^3-`XU5p^_`<-a~uQ<BV@mqMeb)-uq^*l4#sF_V<Bv?Q{I2kk3TRacf77MFP)tiGW>&hfFZ=PmPzJGePqudf<Wd%azs{dQ+*;F*bAsPOJqrr}bF9yq|e$D|Y3E6sw+fAA<&Kdn1C;H7G|{{He$-7R`w%5V(c$M~dcRsbT*yMN7H#`ZXj=#)G0D3#|ezysDH@%W@_SCKe^8y6n+Ha=@1~?Ttcy>f=Oq)T7bjgB8ZLU~srHViEbP(D{qA*ObIa9{xJTVd>|q@XI!z?6UB%a+o{kLko0axbRB32ZEg|&Nlu&|Io$AGQF3b%?w4x^qWJpDEm;`x``e~;>3C};<IgiZzKC%UWT07RXQ0G*Dt5(l2k})UfON5w;1&;*Ta^&3^19qn#~*=#=^x&hlWP}SI;;A(9^Wd{V;CuseJ(zuPQsAxZAoa{7DfAZF9Cg8Q6@VQo`1GyXJ(_U@lLnMcs$|OPHCLgU{(9X<5uX?rtmdpMBEZuCfZgYFURZWmwd7kSFerk6_CC>s5Q@=;akMnZMepdT+z;*P0-{1r4`WqC{UfehRI5PZ03_D*2fb=^&nRSCZw$;428ot5%POz}FFJVM=M0<{T0DPNq=8z{G94zDT81oLYDl4Y$`etn-QK_F2ajWE%KBeN+cKzkB2lH*Oe85cjI3v(np|GrqM~ze)8OUv~sYvOWt%T7H!_Ep3>*z8bne*PMT@IX@wUDZ<!X4d)gK%%&V=p+O2zFdxqJ9fjAl;lIb7A(B~M#BoMj_v#3-AHqqfZaV;T?l0%2gmm#-L|^QR5o~Q?3<u7<F$r@yJQ2}hAi_*^e<~4pGh9MG5CJ<fm(5n}y**gHPJg;cWn_;|IaGz1_fv=HczTEE_nG6u(9kqvThNw-l)^9Cc;Q(W=|36X_oZ>va4Q$5vV1wfJb@KcMOU9_^?qiQ$FG0Zzs|KaP)-uW(O5;T#GEs?B&K%Aw9Ki~FUU9qUc~cgeE;G^2w^ve97qE15Z!~%lE(TKhvu;Tdt(D|!u3jw#QTDv9U1eipGDMN+#e>k#W1}s-nRN*^jSigSIfp~P7iVThDOp`OkbQai0eM)ij$bH7G3_7kQVVqx`<fPM8uN)rYG<m??mpknzQ3ohX&is<Zx;#lnq!b5~>M5caD(Lfj<gj3p&LzH=^W|<DQgJW<7?OgZ8S|w+bWlYU9-j?{cx|ExqV1x#*xf#hSM`K&yzwQmi<`rR2=5?t?zG;-7k(KX3Lcm)-jGsI^yf)pd&17NlG$beR<0bWIiLX+c_MVVsniV88ss<dPH;6(`9M9SO}sIGP#?h#4x_Hr|^Z6or14PE^;@SS?JPy<y)?rM0!#4q~Pc_zKX(S@N8R!h+r{BJ$0l@?yk@SU>dLpXYpRBd>MLWt<5>)r`YutQNIxnuSRm5^PY#le_WlUG)=5s$o?4I-+$^&v7|C7RSEHNL>inDbl#4i~$bA8Qb6NO|-hQG*v#+FwaeKnj-dteX{5Gs5I_9q{B5OhN6TGQM^KqLf_d!+fN)<ZZ;n|DyS_!*V?DwqJ#BdoBPvXP(+o=+(K2UNmS1`7OG@_f~ZMB9|A=Zb=!NT5Wl(_?N*$<(;pf|N3@rH`GQDR@<Y+Jqy5{y9B|Cvk-njq3C@_n;h%-?+02d^^ZOJfX-xWD;AO{Is~zc#F~)w8i_LjzI3)GwNKFw%801#coNQF-Uf1t+06?tPUSHu3z>dz(VOK_a_%ACmAnV&;(;%4|*roCWBq)TvrAR2L6YB(Hsr9>t0W8+<B(7xF0Ny?NB$^jq1l$pmpGtG-<F!PN&n>k?eZh5|ZpZ_))kEFci0YZ@A(u5Y#L-k~LcnU_UCE(pM%mIe5M32o471W6C+J%cm7q7byzrqf>y5m?tT$q#t>*Hsshsn0ufb)(3RBBStQZVh++43;rQu9Dgm-Xf7`{B@T$7NLSFU=aeq|>3sbt4d74s=lG6WH4?Vb_^GlD~|EY_&+RrbPnv=$;T33kGY+ns`>R(E;Amm3F{8{#S{*EZu!UTz#6ofg+g<@zCjKCNr7S2``~_Zz$+UCX~#T&FQ3KiMZt#?%O#;;6e);yS+Rmb|j%NKCk3RQ4OvH)gZQchxKm_aBbqD6(4N$f_$mHZUtsg_CR;qp<GB$vm!{$yb#XeKx`jQ5>xzE1ep}Gu&0=i1NG$9&c8}VUq_w--?B_rkwR6IaNE3Jbf`45v?X-F-jI=YAY7+A?e?euPtd?B@!6OJOYx|baz2U=3(B2Y)Zm<mjrRmwGgMypYg_!-X=!AQ0t9h=6}a=SdDmThMISvgjaPHgk{!uy-nEo#dQyv6xG*jEJGhwUl^BNn%8x@dLxTVk!c|B{;0dlp28)Bs#!TtFljTQ=@?g@i<{2H#U1;0k*WqOEOEuf*hixAiOQ5g&xBS3f9Z!aL<w*`wKYe{VqF;Ffj%oKWrQd%X%S&t0-5AmQB>*%GP%JQ*#&V|VVUKQ=(P-x44r}6Dr~hR=W*yc<^(wpT;rpNP{biGjl749Qf6{y;a)D?tMkCxkCKfd(g4@%ABZ0FpjRr%7sb(5>0T>FBj<=y@dAmQZRc~TjYxHeVp>_zb1kGiaEa;@c()?bv&RSW2HMncQAym}Uu(`*f^wlA!JKE9NvticnC<`-0L@B$%WO?4TUM$Lj)qNzq;JoX)RzcqpX_>#zOFk@mTSA_wtC9tqpy26%qSeMdzo6|!@ZI=JV>C32yYKVlfg)85*UJX@x9z+$+tfEBO64-Ef6mD<$+_mDoNhhQMCDsL2}I8yRw>$8Hthqv53yQi0J$Q9wslKb}2&x^e$Q(70Ic?>nAgS&x0%f&Y$W8;L>0gjO(H{QYvJ>IPPwxwNDZVglR4;D{E4V%l4FsfZBYf=}TLgEdJmBrb)21qE=y%`BRdLKNN&lhu1ZuE#uSiWpv!o<~2KbuKrYxfUD_mLC=bquA5e{*zK*YZ7ROiNb=2<Xs#ZTyF>ciEXed`!KZ<E1vXJ%#|;()tZ*SmrzdAMXVVPSya~Chs*D-jd2`*lW>rnOD~zDrqzs@N4zaRH_$H_X@QgHbj&|~?_->ot*7mIDU5F+$7-EY81E8D~;)wuFZ<7haW*;mF3d&|_j4F_VqJE-MvaGBzz*XqAm9XKetKmy!MX|b{ryXZ0jAiOLMmbg#dMVZ?&PmexOQSi#MT$-5FOKUPad(}%b1^u-IQ(z{*UJ5-w*muBEam+%va<)W^L{ZOg4hg$Ynq4%t9~6L3OWa%78WU@K<{3Fu4g8}fGL9HIkrXo(rmQ$+yC+JBRHQo8|`+Jez$*tR1rkWVC>hUq#oRgDA10JZV*S32-|F2hq59Qi9^Q|am0UfUa64ZZe#~{k&Iai#X@#@%Etnp^BWNY=SZ4(jfXZpB<5^>qb}<8y7hnbYJ1Y_u;ahcte)+)`S1SjF8^&mYjQJis^(LxI^X@F_P0gSifxC{SYLr={!CT3wy3eE?X4~Odw*|F|876y-=f(hMu6=6PF~?q!`K(Y*lWpQv_*MDbC3cBmwVch!`+wVc2MpzIuh`!^9r1Nz*%c?YNF0v+|_mSI&vH!mTl%*L?(iIX_WZ7)5h0iF(jSKT04=KquJPf&Up;1!*v?s?oLCkUJ*jR*?7LwMClTg&F{|`X|}drA|oP9n6v=d0_G?h5^+3W+>L+NA=thoCVq161S%0EGYmgcqnFr47%8(1@X?sa4?yEmor~KvKs^OKVkCRdoMndjKmFUk{u@DlN9dvo)iq`kL+o2ybAxCfWK9xa@iY!esr2s#FbdLUJ-f@VsiPg*&NDx{TKHFZoA6tDCrgRv9U=z06saha7!6uQHoj1iXgGm8C2`r^;$c8kC{8yzdtd&`{|8_K`HFMR000"
public_nb = json.loads(gzip.decompress(base64.b85decode(PUBLIC_NOTEBOOK_B85)).decode("utf-8"))
public_code = [cell["source"] for cell in public_nb["cells"] if cell["cell_type"] == "code"]
assert len(public_code) == 5, "Unexpected public notebook layout"

for source, filename in zip(public_code[1:4], ("pv.py", "pv_fp.py", "casmi_engine.py")):
    magic, body = source.split("\n", 1)
    assert magic == f"%%writefile {filename}"
    (WORK / filename).write_text(body)

exec(compile(public_code[0], "<public_setup>", "exec"), globals())
print("Engine:", ", ".join(path.name for path in WORK.glob("*.py")))

In [ ]:
'''Set the within-molecule blend.

0.65 follows the public reference: the first ranker uses fingerprint evidence;
the second ranker deliberately excludes it to soften library-leakage risk.
'''
BLEND_PUBLIC_RANKER = 0.65
assert 0.0 <= BLEND_PUBLIC_RANKER <= 1.0

In [ ]:
'''Run the only expensive stage.

This writes the recommended blend as submission.csv plus the two ablations:
submission_pv.csv and submission_ours.csv.
'''
import os
import time

if BLEND_PUBLIC_RANKER == 0.65:
    exec(compile(public_code[4], "<public_run>", "exec"), globals())
else:
    import casmi_engine as E
    fp_models = sorted(glob.glob("/kaggle/input/**/fp_*.pt", recursive=True))
    subs, recs = E.main(
        os.path.join(COMP, "test.parquet"),
        os.path.join(COMP, "train.parquet"),
        os.path.join(COMP, "sample_submission.csv"),
        find("sim_rank_rows_nofp.npz"),
        find("rank_train.npz"),
        fp_models,
        workers=os.cpu_count(),
        w_pv=BLEND_PUBLIC_RANKER,
    )
    for name, frame in subs.items():
        frame.to_csv("submission.csv" if name == "blend" else f"submission_{name}.csv", index=False)

In [ ]:
'''Verify the exact Kaggle scoring boundary before saving a version.'''
import pandas as pd

submission = pd.read_csv("submission.csv")
template = pd.read_csv(os.path.join(COMP, "sample_submission.csv"))

assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert submission.columns.tolist() == template.columns.tolist()
assert submission["molecule_id"].tolist() == template["molecule_id"].tolist()
assert submission["molecule_id"].is_unique
assert submission["smiles"].notna().all()
assert (submission["smiles"].str.strip() != "").all()
assert (submission["smiles"].str.split(";").str.len() <= 25).all()

print(f"Ready: {len(submission):,} rows; top-{submission.smiles.str.split(';').str.len().max()} max")
display(submission.head(3))

In [ ]:
'''Print the three submission variants.

Submit each one once; only then tune BLEND_PUBLIC_RANKER around the winner.
'''
print("Files:", [path.name for path in WORK.glob("submission*.csv")])